# Puneri · item photos

Generates a real-looking product photo of every item in the jhola (inventory), cut out on a transparent background, all in the same studio light so they sit together. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 10–15 minutes. At the end a zip downloads: send it back and the game uses the photos (anything without one keeps its painting).

Want a different look for one item? Edit its line in `ITEMS` and run the last two cells again.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors 'rembg[gpu]' onnxruntime-gpu pillow

In [ ]:
ITEMS = {
 "water": "a 1 litre clear plastic packaged drinking water bottle with a blue cap and a blue label, Indian mineral water",
 "tiffin": "a three-tier stainless steel Indian tiffin box with a carrying handle, polished steel",
 "vadapav": "a Mumbai vada pav: a golden fried potato vada in a soft pav bun with green chutney and one fried green chilli, on brown paper",
 "biscuit": "a classic Indian glucose biscuit packet, yellow wrapper with a red band, a few biscuits beside it",
 "banana": "a small bunch of three ripe yellow Indian bananas (elaichi bananas)",
 "chikki": "a bar of Lonavala peanut chikki, golden jaggery brittle with peanuts, broken in two",
 "medkit": "a small white first aid kit box with a red cross and a handle",
 "painkiller": "a silver blister strip of white paracetamol tablets",
 "ors": "an orange ORS oral rehydration salts sachet, Indian pharmacy packet",
 "energy": "a slim aluminium energy drink can, dark blue with a yellow lightning bolt",
 "coldcoffee": "a tall clear plastic cup of Indian cold coffee with cream on top and a green straw",
 "bakarwadi": "a pile of Pune bakarwadi, small crisp spiral fried savoury rolls",
 "barfi": "an open red sweet box with diamond-shaped kaju katli barfi with silver leaf",
 "shrewsbury": "a round dark blue tin of shortbread Shrewsbury biscuits with the lid off, golden butter biscuits inside",
 "gajra": "a gajra, a circular string of fresh white jasmine flowers",
 "book": "a hardbound city guide book with a maroon cloth cover and gold lettering",
 "lottery": "an Indian state lottery ticket, pastel coloured paper with printed numbers",
 "torch": "a black metal LED flashlight torch, switched on, glowing lens",
 "beer": "a brown glass beer bottle, 650 ml, with a red and white label, cold with droplets",
 "phone": "a modern black smartphone, screen on with a colourful wallpaper, three quarter view",
 "keys": "a bunch of vehicle keys on a ring: a two-wheeler key and an auto rickshaw key with a red rubber tag"
}
STYLE = ", studio product photograph, single object centred, three quarter view, soft key light from top left, gentle rim light, photorealistic, sharp focus, 85mm lens, plain light grey seamless background, no text, no watermark"
NEG = 'text, letters, logo, watermark, hands, people, blurry, cartoon, illustration, drawing, 3d render, low quality, multiple objects, cropped'

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')
pipe.enable_attention_slicing()

In [ ]:
import os, io, zipfile
from rembg import remove, new_session
from PIL import Image
os.makedirs('items', exist_ok=True)
sess = new_session('isnet-general-use')
for i, (k, p) in enumerate(ITEMS.items()):
    g = torch.Generator('cuda').manual_seed(1000 + i)
    im = pipe(prompt=p + STYLE, negative_prompt=NEG, num_inference_steps=30, guidance_scale=6.5, width=1024, height=1024, generator=g).images[0]
    cut = remove(im, session=sess)                       # transparent background
    bb = cut.getbbox(); cut = cut.crop(bb) if bb else cut
    s = max(cut.size); sq = Image.new('RGBA', (s, s), (0, 0, 0, 0)); sq.paste(cut, ((s - cut.size[0]) // 2, (s - cut.size[1]) // 2))
    sq.resize((512, 512), Image.LANCZOS).save(f'items/{k}.webp', 'WEBP', quality=90)
    print('done', k)
    display(sq.resize((160, 160)))

In [ ]:
with zipfile.ZipFile('puneri_items.zip', 'w') as z:
    for f in sorted(os.listdir('items')): z.write('items/' + f, 'items/' + f)
from google.colab import files
files.download('puneri_items.zip')